# FOLHA DE COLA — EXTRAÇÃO E ANÁLISE DE DADOS
### Prof. Pestana — FGV 2026.2 | Aulas 1 a 13

**Ordem das secoes:**
1. Pandas — inspeção, filtro, limpeza (Aulas 4-5)
2. Hashtags — explodir e agrupar (Aula 5)
3. Matplotlib — graficos de barras, linhas e dispersão (Aula 6)
4. Web scraping — requests + BeautifulSoup (Aula 8)
5. Playwright — paginas dinamicas (Aula 9, estrutura)
6. Limpeza e pipeline (Aula 10)
7. Machine Learning — Regressão (Aula 11)
8. Machine Learning — Classificação (Aula 12)
9. Machine Learning — Clusterização KMeans (Aula 13)

**Rode a celula de DADOS primeiro. Depois rode qualquer secao.**

---
## DADOS DE EXEMPLO — rode essa celula primeiro!
Cria tres bases embutidas que vao ser usadas em todo o notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(42)
np.random.seed(42)

# -------------------------------------------------------
# BASE 1: df_posts — simula exportacao do Zeeschuimer
# (TikTok/Instagram com curtidas, comentarios, hashtags)
# -------------------------------------------------------
autores = ['@ana_politica','@joao_news','@maria_dados','@carlos_tec','@lucia_social']
hashtags_pool = [
    'politica,eleicoes,brasil',
    'tecnologia,ia,dados',
    'economia,inflacao',
    'politica,congresso',
    'ia,machine_learning,python',
    'brasil,noticias',
    'eleicoes,votacao,politica',
    'dados,analise,python',
    '',
    'economia,brasil,inflacao'
]
n = 200
df_posts = pd.DataFrame({
    'author': random.choices(autores, k=n),
    'likes': np.random.exponential(scale=500, size=n).astype(int) + 10,
    'comments': np.random.exponential(scale=50, size=n).astype(int) + 1,
    'shares': np.random.exponential(scale=30, size=n).astype(int),
    'plays': np.random.exponential(scale=5000, size=n).astype(int) + 100,
    'hashtags': random.choices(hashtags_pool, k=n),
    'body': ['Post sobre ' + random.choice(['politica','tecnologia','economia','dados','IA']) + '. ' * random.randint(1,10) for _ in range(n)],
    'timestamp': pd.date_range('2024-01-01', periods=n, freq='12h')
})
# Adiciona duplicata intencional
df_posts = pd.concat([df_posts, df_posts.iloc[:5]], ignore_index=True)
df_posts['taxa_engajamento'] = (df_posts['likes'] + df_posts['comments']) / (df_posts['plays'] + 1)
print("df_posts criado:", df_posts.shape)
df_posts.head(3)

In [ ]:
# -------------------------------------------------------
# BASE 2: df_livros — simula scraping de books.toscrape.com
# -------------------------------------------------------
categorias = ['Mystery','Science','History','Travel','Fiction','Nonfiction']
df_livros = pd.DataFrame({
    'titulo': ['Livro ' + str(i) for i in range(1, 51)],
    'preco': np.round(np.random.uniform(10, 60, 50), 2),
    'avaliacao': random.choices([1,2,3,4,5], weights=[5,10,20,35,30], k=50),
    'disponivel': random.choices(['In stock','Out of stock'], weights=[80,20], k=50),
    'categoria': random.choices(categorias, k=50),
})
print("df_livros criado:", df_livros.shape)
df_livros.head(3)

In [ ]:
# -------------------------------------------------------
# BASE 3: df_clientes — simula dados para ML (regressao, classificacao, cluster)
# -------------------------------------------------------
n = 300
df_clientes = pd.DataFrame({
    'idade': np.random.randint(18, 70, n),
    'renda_mensal': np.random.exponential(3000, n) + 1500,
    'gasto_mensal': np.random.exponential(800, n) + 200,
    'num_compras': np.random.poisson(5, n),
    'tempo_cliente_meses': np.random.randint(1, 120, n),
    'cidade': random.choices(['SP','RJ','BH','POA','Recife'], weights=[35,25,15,15,10], k=n),
})
df_clientes['churn'] = (
    (df_clientes['gasto_mensal'] < 300) &
    (df_clientes['num_compras'] < 3)
).astype(int)
print("df_clientes criado:", df_clientes.shape)
df_clientes.head(3)

---
## 1. PANDAS — inspeção, filtro e limpeza
Usando df_posts (dados de redes sociais)

In [ ]:
# INSPECIONAR A BASE — sempre começa por aqui
print("Tamanho:", df_posts.shape)        # (linhas, colunas)
print()
print("Colunas e tipos:")
print(df_posts.dtypes)                   # object = texto | int/float = numero
print()
print("Valores ausentes por coluna:")
print(df_posts.isna().sum())
print()
print("Duplicatas:", df_posts.duplicated().sum())

In [ ]:
# REMOVER DUPLICATAS
df = df_posts.drop_duplicates().copy()
print("Linhas depois de remover duplicatas:", len(df))

# FILTRAR linhas
df_ativos = df[df['plays'] > 100].copy()          # so posts com mais de 100 plays
df_autor = df[df['author'] == '@ana_politica']     # posts de um autor especifico

# CRIAR NOVA COLUNA
df['tamanho_legenda'] = df['body'].str.len()       # tamanho do texto do post

# ORDENAR
top5 = df.sort_values('likes', ascending=False).head(5)
print(top5[['author','likes','plays','hashtags']])

In [ ]:
# ESTATISTICA DESCRITIVA
print(df[['likes','comments','shares','plays','taxa_engajamento']].describe())

print()
print("Media de likes:", df['likes'].mean().round(2))
print("Mediana de likes:", df['likes'].median())

# AGRUPAR POR AUTOR — media de engajamento por autor
resumo = df.groupby('author').agg(
    total_posts=('likes', 'count'),
    media_likes=('likes', 'mean'),
    media_taxa_eng=('taxa_engajamento', 'mean')
).reset_index()
resumo['media_likes'] = resumo['media_likes'].round(0)
resumo['media_taxa_eng'] = resumo['media_taxa_eng'].round(4)
print(resumo.sort_values('media_taxa_eng', ascending=False))

---
## 2. HASHTAGS — o padrao dropna → split → explode → strip → groupby
Esse padrao aparece em todas as aulas 5, 6 e 7. Decore ele.

In [ ]:
# PADRAO COMPLETO: explodir hashtags separadas por virgula
df_hashtags = df.dropna(subset=['hashtags']).copy()      # remove linhas sem hashtag
df_hashtags = df_hashtags[df_hashtags['hashtags'] != ''] # remove hashtag vazia
df_hashtags['hashtags'] = df_hashtags['hashtags'].str.split(',')  # string -> lista
df_hashtags = df_hashtags.explode('hashtags')             # uma hashtag por linha
df_hashtags['hashtags'] = df_hashtags['hashtags'].str.strip()     # remove espacos

print("Linhas apos explodir:", len(df_hashtags))
print("Hashtags unicas:", df_hashtags['hashtags'].nunique())
print()
print("Top 10 mais frequentes:")
print(df_hashtags['hashtags'].value_counts().head(10))

In [ ]:
# TABELA-RESUMO POR HASHTAG: frequencia + engajamento medio
tabela_hashtags = df_hashtags.groupby('hashtags').agg(
    total_posts=('likes', 'count'),
    engajamento_medio=('taxa_engajamento', 'mean')
).reset_index()
tabela_hashtags['engajamento_medio'] = tabela_hashtags['engajamento_medio'].round(4)
tabela_hashtags = tabela_hashtags.sort_values('total_posts', ascending=False)

# Salvar em CSV
tabela_hashtags.to_csv('hashtags_resumo.csv', index=False)
print("Top 10 hashtags por frequencia:")
print(tabela_hashtags.head(10))

---
## 3. MATPLOTLIB — graficos de barras, linhas e dispersão
Usa matplotlib (nao altair). Padrao: fig, ax = plt.subplots() → ax.bar/plot/scatter → titulos → plt.show()

In [ ]:
# GRAFICO DE BARRAS — top 10 hashtags por frequencia
top10 = tabela_hashtags.head(10)

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(top10['hashtags'], top10['total_posts'], color='#3b6ea5')

ax.set_title('Top 10 hashtags por frequencia de postagens')    # titulo obrigatorio
ax.set_xlabel('Hashtag')                                        # rotulo eixo X
ax.set_ylabel('Total de posts')                                 # rotulo eixo Y com unidade
ax.tick_params(axis='x', rotation=45)                           # gira os rotulos
fig.text(0.01, -0.04, 'Fonte: df_posts simulado', fontsize=8, color='gray')  # fonte dos dados

fig.tight_layout()
plt.show()

In [ ]:
# GRAFICO DE LINHA — evolucao de posts ao longo do tempo
df_tempo = df.copy()
df_tempo['data'] = df_tempo['timestamp'].dt.date
posts_por_dia = df_tempo.groupby('data')['likes'].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(posts_por_dia['data'], posts_por_dia['likes'], marker='o', color='#e07b39')

ax.set_title('Media de curtidas por dia')
ax.set_xlabel('Data')
ax.set_ylabel('Media de curtidas')
ax.tick_params(axis='x', rotation=45)
fig.text(0.01, -0.06, 'Fonte: df_posts simulado', fontsize=8, color='gray')

fig.tight_layout()
plt.show()

In [ ]:
# GRAFICO DE DISPERSAO — plays vs likes (escala log para ver melhor)
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(df['plays'], df['likes'], alpha=0.4, color='#5b9a5e')

ax.set_title('Relacao entre visualizacoes (plays) e curtidas')
ax.set_xlabel('Plays (visualizacoes)')
ax.set_ylabel('Curtidas')
ax.set_xscale('log')   # escala logaritmica quando os valores variam muito
fig.text(0.01, -0.04, 'Fonte: df_posts simulado', fontsize=8, color='gray')

fig.tight_layout()
plt.show()

---
## 4. WEB SCRAPING — requests + BeautifulSoup
Usa para paginas estaticas (HTML simples, sem JavaScript).

In [ ]:
# ESTRUTURA COMPLETA DE SCRAPING
import requests
from bs4 import BeautifulSoup
import csv

url = 'https://books.toscrape.com/'    # <-- troca pela URL da prova

resposta = requests.get(url, headers={'User-Agent': 'FGV-ExtratorDados/1.0'})

if resposta.status_code == 200:
    resposta.encoding = resposta.apparent_encoding  # corrige encoding
    html = resposta.text
    print("OK! Tamanho do HTML:", len(html), "caracteres")
else:
    html = None
    print("Erro:", resposta.status_code, "— confira a URL e a conexao")

In [ ]:
# PARSEAR O HTML E EXTRAIR DADOS
# Obs: books.toscrape.com e um site de treino publico para scraping

sopa = BeautifulSoup(html, 'html.parser')

itens_extraidos = []

for item in sopa.find_all('article', {'class': 'product_pod'}):
    # .find() acha o primeiro elemento daquela tag/classe DENTRO do item
    titulo = item.find('h3').find('a')['title']         # pega atributo title
    preco = item.find('p', {'class': 'price_color'}).get_text(strip=True)
    avaliacao = item.find('p', {'class': 'star-rating'})['class'][1]  # "Three", "Four"...
    disponivel = item.find('p', {'class': 'availability'}).get_text(strip=True)

    itens_extraidos.append({
        'titulo': titulo,
        'preco': preco,
        'avaliacao': avaliacao,
        'disponivel': disponivel,
    })

print(f"Extraidos: {len(itens_extraidos)} livros")
print(itens_extraidos[:3])

In [ ]:
# SALVAR EM CSV
with open('livros_extraidos.csv', 'w', newline='', encoding='utf-8') as f:
    colunas = ['titulo', 'preco', 'avaliacao', 'disponivel']
    escritor = csv.DictWriter(f, fieldnames=colunas)
    escritor.writeheader()               # escreve a linha de cabecalho
    escritor.writerows(itens_extraidos)  # escreve todas as linhas de uma vez

print("CSV salvo! Linhas:", len(itens_extraidos))

# LER DE VOLTA PARA CONFERIR
df_scraped = pd.read_csv('livros_extraidos.csv')
df_scraped.head()

---
## 4b. SCRAPING — percorrer multiplas paginas (paginacao)
Mesmo padrao, mas com loop de paginas.

In [ ]:
# EXEMPLO DE PAGINACAO — books.toscrape.com tem 50 paginas
import time

todos_livros = []

for pagina in range(1, 4):  # <-- troca o 4 pelo numero real de paginas
    if pagina == 1:
        url_pag = 'https://books.toscrape.com/'
    else:
        url_pag = f'https://books.toscrape.com/catalogue/page-{pagina}.html'

    resp = requests.get(url_pag, headers={'User-Agent': 'FGV-ExtratorDados/1.0'})
    if resp.status_code != 200:
        print(f"Pagina {pagina} falhou com status {resp.status_code}")
        continue

    resp.encoding = resp.apparent_encoding
    sopa_pag = BeautifulSoup(resp.text, 'html.parser')

    for item in sopa_pag.find_all('article', {'class': 'product_pod'}):
        titulo = item.find('h3').find('a')['title']
        preco = item.find('p', {'class': 'price_color'}).get_text(strip=True)
        todos_livros.append({'titulo': titulo, 'preco': preco, 'pagina': pagina})

    print(f"Pagina {pagina}: ok, total acumulado = {len(todos_livros)}")
    time.sleep(1)  # espera 1 segundo entre requisicoes (boa pratica!)

print("Total extraido:", len(todos_livros))
pd.DataFrame(todos_livros).head()

---
## 5. PLAYWRIGHT — paginas dinamicas (JavaScript)
Use quando requests+BS4 nao funciona (pagina precisa de JavaScript para carregar).
ATENCAO: Playwright NAO roda dentro do Jupyter — rode no terminal com: uv run script.py

In [ ]:
# ESTRUTURA DO SCRIPT PLAYWRIGHT (salva como arquivo .py e roda no terminal)
# Cole esse codigo num arquivo chamado coletar.py e rode:
#   uv run coletar.py
# ou, com .venv ativado:
#   python coletar.py

codigo_playwright = """
from playwright.sync_api import sync_playwright
import csv, time

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)   # headless=True = sem janela
    page = browser.new_page()

    page.goto('https://exemplo.com')              # <-- URL da pagina alvo
    page.wait_for_selector('[data-testid="item"]') # espera elemento aparecer

    # CLICAR EM BOTAO (ex: "Carregar mais")
    for i in range(5):
        botao = page.query_selector('[data-testid="load-more-button"]')
        if botao:
            botao.scroll_into_view_if_needed()
            botao.click()
            time.sleep(1.5)
            print(f"Clique {i+1}: ok")

    # EXTRAIR ELEMENTOS DA PAGINA
    cards = page.query_selector_all('[data-testid="story-item"]')
    itens = []
    for card in cards:
        titulo = card.text_content().strip()
        link = card.query_selector('a')
        href = link.get_attribute('href') if link else ''
        itens.append({'titulo': titulo, 'url': href})

    # SALVAR SCREENSHOT
    page.screenshot(path='dados/screenshot.png')

    browser.close()

# Salvar em CSV
with open('dados/resultado.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['titulo', 'url'])
    writer.writeheader()
    writer.writerows(itens)

print(f"Coletados {len(itens)} itens")
"""

print("Cole o codigo acima num arquivo .py e rode no terminal.")
print()
print("Comandos Playwright mais usados:")
print("  page.goto(url)                    → navega para a URL")
print("  page.wait_for_selector('...')     → espera elemento aparecer")
print("  page.query_selector_all('...')    → lista todos os elementos")
print("  elemento.click()                  → clica")
print("  elemento.scroll_into_view_if_needed() → rola ate o elemento")
print("  elemento.text_content()           → texto do elemento")
print("  elemento.get_attribute('href')    → valor do atributo href")

---
## 6. LIMPEZA E PIPELINE (Aula 10)
Sequencia obrigatoria: ler bruto → copiar → limpar → validar → salvar em processed/

In [ ]:
# PIPELINE DE LIMPEZA COMPLETO
# Simula um CSV bruto com problemas comuns
dados_brutos = {
    'titulo': ['  Livro A ', 'livro b', 'LIVRO C', 'Livro A ', None, 'Livro D'],
    'preco': ['£12.50', '£8.99', 'invalido', '£15.00', '£20.00', '£8.99'],
    'data_publicacao': ['01/03/2023', '2023-05-15', '15-07-2023', 'invalida', '2023-09-01', '01/03/2023'],
    'avaliacao': ['One', 'Three', 'Five', 'Two', 'Four', 'Three'],
}
df_bruto = pd.DataFrame(dados_brutos)
print("BRUTO:")
print(df_bruto)
print()

In [ ]:
# LIMPEZA PASSO A PASSO
df = df_bruto.copy()   # NUNCA altere o bruto diretamente

# 1. Remover duplicatas
print("Duplicatas antes:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicatas depois:", df.duplicated().sum())

# 2. Remover linhas sem titulo (coluna obrigatoria)
df = df.dropna(subset=['titulo'])

# 3. Normalizar texto
df['titulo'] = df['titulo'].str.strip()   # remove espacos do inicio/fim
df['titulo'] = df['titulo'].str.title()   # Primeira Letra Maiuscula

# 4. Limpar e converter preco (remove o simbolo £, converte para numero)
df['preco'] = df['preco'].str.replace('£', '', regex=False)
df['preco'] = pd.to_numeric(df['preco'], errors='coerce')  # invalido vira NaN

# 5. Converter data (formatos diferentes → errors='coerce' transforma invalido em NaT)
df['data_publicacao'] = pd.to_datetime(df['data_publicacao'], dayfirst=True, errors='coerce')

print()
print("LIMPO:")
print(df)
print()
print("Ausentes apos limpeza:")
print(df.isna().sum())

In [ ]:
# SALVAR RESULTADO LIMPO
# Convencao de pastas:
#   dados/raw/      → arquivo bruto (nunca sobrescreva!)
#   dados/processed/ → arquivo limpo (salva aqui)

df.to_csv('livros_processados.csv', index=False)
print("Salvo em livros_processados.csv")
print("Linhas que entraram:", len(df_bruto))
print("Linhas que saíram:", len(df))

---
## 7. MACHINE LEARNING — REGRESSAO (Aula 11)
Prever um NUMERO (ex: taxa de engajamento, preco, plays).

Esqueleto: definir X e y → split treino/teste → fit → predict → medir MAE e R2 → comparar com modelo bobo

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ALVO (o que queremos prever)
y = df_clientes['gasto_mensal']

# FEATURES (o que o modelo vai usar para prever — NUNCA coloque o alvo aqui!)
X = pd.DataFrame(index=df_clientes.index)
X['idade'] = df_clientes['idade']
X['renda_mensal'] = df_clientes['renda_mensal']
X['num_compras'] = df_clientes['num_compras']
X['tempo_cliente_meses'] = df_clientes['tempo_cliente_meses']
# Variavel categorica vira dummy (0 ou 1)
X = pd.get_dummies(X.join(df_clientes[['cidade']]), columns=['cidade'], drop_first=True)

print("Features usadas:", X.columns.tolist())
print("X shape:", X.shape, "| y shape:", y.shape)

In [ ]:
# SPLIT TREINO/TESTE
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Treino:", len(X_treino), "linhas | Teste:", len(X_teste), "linhas")

# MODELO BOBO — preve sempre a media do treino (referencia minima)
bobo = np.full(len(y_teste), y_treino.mean())
mae_bobo = mean_absolute_error(y_teste, bobo)
r2_bobo = r2_score(y_teste, bobo)
print(f"Modelo bobo — MAE: {mae_bobo:.2f} | R2: {r2_bobo:.2f}")

In [ ]:
# REGRESSAO LINEAR
modelo_lin = LinearRegression()
modelo_lin.fit(X_treino, y_treino)
prev_lin = modelo_lin.predict(X_teste)

mae_lin = mean_absolute_error(y_teste, prev_lin)
r2_lin = r2_score(y_teste, prev_lin)
print(f"Regressao Linear — MAE: {mae_lin:.2f} | R2: {r2_lin:.2f}")

# ARVORE DE REGRESSAO
modelo_arv = DecisionTreeRegressor(max_depth=5, random_state=42)
modelo_arv.fit(X_treino, y_treino)
prev_arv = modelo_arv.predict(X_teste)

mae_arv = mean_absolute_error(y_teste, prev_arv)
r2_arv = r2_score(y_teste, prev_arv)
print(f"Arvore de Regressao — MAE: {mae_arv:.2f} | R2: {r2_arv:.2f}")

print()
print("Resumo:")
print(f"  Modelo bobo:    MAE={mae_bobo:.2f} | R2={r2_bobo:.2f}")
print(f"  Linear:         MAE={mae_lin:.2f} | R2={r2_lin:.2f}")
print(f"  Arvore:         MAE={mae_arv:.2f} | R2={r2_arv:.2f}")
print()
print("MAE = erro medio na unidade do alvo. Menor e melhor.")
print("R2  = 1.0 e perfeito | 0.0 = igual ao bobo | <0 = pior que bobo")

In [ ]:
# COEFICIENTES DA REGRESSAO LINEAR (o que cada feature influencia)
coef = pd.DataFrame({
    'feature': X.columns,
    'coeficiente': modelo_lin.coef_
}).sort_values('coeficiente', key=abs, ascending=False)
print("Coeficientes (positivo = aumenta o gasto, negativo = diminui):")
print(coef)
print()
print("ATENCAO: coeficiente descreve ASSOCIACAO nos dados, nao causa!")

In [ ]:
# QUANDO O ALVO TEM CAUDA LONGA (ex: plays, curtidas, contagens)
# Use log1p para comprimir — desfaz com expm1
y_log = np.log1p(y)  # log(1 + x) — evita log(0)
X_treino2, X_teste2, y_treino2, y_teste2 = train_test_split(y_log, y_log, test_size=0.25, random_state=42)

modelo_log = LinearRegression()
# Treina no log
X_tr, X_te, y_tr, y_te = train_test_split(X, y_log, test_size=0.25, random_state=42)
modelo_log.fit(X_tr, y_tr)
prev_log_scale = modelo_log.predict(X_te)
prev_original = np.expm1(prev_log_scale)  # desfaz o log para voltar a escala original

mae_log = mean_absolute_error(np.expm1(y_te), prev_original)
print(f"Regressao com log1p — MAE na escala original: {mae_log:.2f}")

**Regras de ouro da regressao:**
- Nunca coloque o alvo (y) dentro do X — isso e VAZAMENTO
- Sempre compara o modelo com o modelo bobo. Se nao bate, nao aprendeu nada
- Mede o erro no TESTE, nunca no treino
- Coeficiente descreve associacao, nao causa

---
## 8. MACHINE LEARNING — CLASSIFICACAO (Aula 12)
Prever uma CATEGORIA (ex: vai viralizar ou nao, churn ou nao).

Esqueleto: criar rotulo → split com stratify → fit → predict/predict_proba → matriz de confusao → precisao/recall/F1

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# CRIAR O ROTULO (decisao metodologica — documente o corte escolhido!)
corte = df_clientes['gasto_mensal'].quantile(0.75)  # top 25% de gastos
y_class = (df_clientes['gasto_mensal'] >= corte).astype(int)
print(f"Corte escolhido: R${corte:.2f}")
print(f"Positivos (gasto alto): {y_class.mean():.1%} dos clientes")

# FEATURES (mesmas da regressao, sem o alvo)
X_class = pd.DataFrame(index=df_clientes.index)
X_class['idade'] = df_clientes['idade']
X_class['renda_mensal'] = df_clientes['renda_mensal']
X_class['num_compras'] = df_clientes['num_compras']
X_class['tempo_cliente_meses'] = df_clientes['tempo_cliente_meses']

In [ ]:
# SPLIT COM STRATIFY — mantem a proporcao de 0 e 1 nos dois lados
X_tr, X_te, y_tr, y_te = train_test_split(
    X_class, y_class, test_size=0.25, random_state=42, stratify=y_class
)
print("Proporcao positivos no treino:", y_tr.mean().round(3))
print("Proporcao positivos no teste:", y_te.mean().round(3))

In [ ]:
# REGRESSAO LOGISTICA
modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_tr, y_tr)
decisao = modelo_lr.predict(X_te)

# MATRIZ DE CONFUSAO
cm = confusion_matrix(y_te, decisao)
print("Matriz de confusao:")
print("          Previu 0  Previu 1")
print(f"  Real 0:    {cm[0][0]:4d}      {cm[0][1]:4d}   (VN=acertou negativo, FP=falso alarme)")
print(f"  Real 1:    {cm[1][0]:4d}      {cm[1][1]:4d}   (FN=perdeu positivo, VP=acertou positivo)")
print()
print(f"Acuracia:  {accuracy_score(y_te, decisao):.3f}  — % de acertos totais (engana com classe rara!)")
print(f"Precisao:  {precision_score(y_te, decisao, zero_division=0):.3f}  — dos que chamou de 1, quantos eram 1 de verdade")
print(f"Recall:    {recall_score(y_te, decisao, zero_division=0):.3f}  — dos que eram 1, quantos o modelo pegou")
print(f"F1-score:  {f1_score(y_te, decisao, zero_division=0):.3f}  — equilibrio entre precisao e recall")

In [ ]:
# MEXER NO THRESHOLD (corte de decisao)
probabilidades = modelo_lr.predict_proba(X_te)[:, 1]  # probabilidade de ser 1

# Com threshold 0.3 (mais sensivel — pega mais positivos, mas falha mais)
decisao_03 = (probabilidades >= 0.30).astype(int)
print("Threshold 0.30:")
print(f"  Recall: {recall_score(y_te, decisao_03, zero_division=0):.3f} | Precisao: {precision_score(y_te, decisao_03, zero_division=0):.3f}")

# Com threshold padrao 0.5
print(f"Threshold 0.50:")
print(f"  Recall: {recall_score(y_te, decisao, zero_division=0):.3f} | Precisao: {precision_score(y_te, decisao, zero_division=0):.3f}")

In [ ]:
# ARVORE DE CLASSIFICACAO
modelo_arv_c = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced')
modelo_arv_c.fit(X_tr, y_tr)
decisao_arv = modelo_arv_c.predict(X_te)

print("Arvore de Classificacao:")
print(f"  F1: {f1_score(y_te, decisao_arv, zero_division=0):.3f}")
print(f"  Recall: {recall_score(y_te, decisao_arv, zero_division=0):.3f}")

# IMPORTANCIA DAS FEATURES
importancias = pd.DataFrame({
    'feature': X_class.columns,
    'importancia': modelo_arv_c.feature_importances_
}).sort_values('importancia', ascending=False)
print()
print("Importancia das features (o que o modelo mais usou):")
print(importancias)

**Guia rapido — quando usar cada metrica:**

| Metrica | O que mede | Quando importa |
|---|---|---|
| Acuracia | % de acertos totais | So quando as classes sao equilibradas |
| Precisao | Dos que chamou de 1, quantos eram 1 | Quando falso alarme e caro |
| Recall | Dos que eram 1, quantos pegou | Quando nao pode perder um positivo |
| F1 | Equilibrio entre precisao e recall | Quando as duas importam |

**Regras:**
- Rotulo e decisao metodologica — documente o corte e o motivo
- Nunca avalie so pela acuracia com classe rara
- threshold 0.5 nao e sagrado — ajuste conforme o objetivo

---
## 9. MACHINE LEARNING — CLUSTERIZACAO KMEANS (Aula 13)
Agrupa dados sem rotulo. Encontra grupos parecidos.

Esqueleto: selecionar colunas numericas → padronizar (StandardScaler) → escolher k (cotovelo+silhueta) → fit_predict → interpretar grupos

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# SELECIONAR COLUNAS NUMERICAS
colunas_cluster = ['idade', 'renda_mensal', 'gasto_mensal', 'num_compras', 'tempo_cliente_meses']
X_cl = df_clientes[colunas_cluster].copy()

# PADRONIZAR — OBRIGATORIO antes de clusterizar
# Sem isso, coluna com escala maior (ex: renda em R$) domina tudo
X_pad = StandardScaler().fit_transform(X_cl)
print("Dados padronizados. Shape:", X_pad.shape)
print("Media de cada coluna apos padronizacao (deve ser ~0):", X_pad.mean(axis=0).round(2))

In [ ]:
# ESCOLHER O MELHOR K — curva do cotovelo + silhueta
inercias = []
silhuetas = []
ks = range(2, 9)

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pad)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_pad, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Curva do cotovelo — procura o "cotovelo" onde a queda desacelera
ax1.plot(ks, inercias, marker='o', color='#3b6ea5')
ax1.set_title('Curva do Cotovelo')
ax1.set_xlabel('Numero de clusters (k)')
ax1.set_ylabel('Inercia (soma das distancias)')

# Curva de silhueta — maior silhueta = grupos mais bem separados
ax2.plot(ks, silhuetas, marker='o', color='#e07b39')
ax2.set_title('Silhueta por k')
ax2.set_xlabel('Numero de clusters (k)')
ax2.set_ylabel('Silhueta (0 a 1)')

fig.tight_layout()
plt.show()

melhor_k = ks[silhuetas.index(max(silhuetas))]
print(f"Melhor k pela silhueta: {melhor_k} (silhueta={max(silhuetas):.3f})")

In [ ]:
# APLICAR O KMEANS COM O K ESCOLHIDO
k_escolhido = 3   # <-- troca pelo k que fez sentido no cotovelo/silhueta

kmeans = KMeans(n_clusters=k_escolhido, n_init=10, random_state=42)
df_clientes['cluster'] = kmeans.fit_predict(X_pad)

print("Tamanho de cada cluster:")
print(df_clientes['cluster'].value_counts().sort_index())

In [ ]:
# INTERPRETAR OS CLUSTERS — perfil medio de cada grupo
perfil = df_clientes.groupby('cluster')[colunas_cluster].mean().round(2)
print("Perfil medio por cluster:")
print(perfil)
print()

# Dar nomes aos clusters com base no perfil
# (os numeros 0/1/2 sao arbitrarios — leia o perfil e nomeie)
mapa_nomes = {0: 'Grupo A', 1: 'Grupo B', 2: 'Grupo C'}  # <-- muda conforme o perfil
df_clientes['cluster_nome'] = df_clientes['cluster'].map(mapa_nomes)
print("Distribuicao:")
print(df_clientes['cluster_nome'].value_counts())

In [ ]:
# VISUALIZAR OS CLUSTERS — reduz para 2 dimensoes com PCA para poder plotar
pca = PCA(n_components=2)
coords = pca.fit_transform(X_pad)

fig, ax = plt.subplots(figsize=(8, 6))
cores = ['#3b6ea5', '#e07b39', '#5b9a5e', '#c46e6e', '#8b5ea5']

for cluster_id in sorted(df_clientes['cluster'].unique()):
    mask = df_clientes['cluster'] == cluster_id
    ax.scatter(coords[mask, 0], coords[mask, 1],
               label=mapa_nomes[cluster_id],
               alpha=0.6, s=50, color=cores[cluster_id])

ax.set_title('Clusters visualizados (PCA 2D)')
ax.set_xlabel('Componente principal 1')
ax.set_ylabel('Componente principal 2')
ax.legend()
fig.tight_layout()
plt.show()

**Regras de ouro da clusterizacao:**
- SEMPRE padronizar com StandardScaler antes do KMeans
- Justifique a escolha de k (cotovelo + silhueta)
- Os numeros 0/1/2 sao arbitrarios — mapeie para nomes pela leitura do perfil
- Um cluster so vale quando faz sentido descrito em palavras
- A escolha das features muda o resultado — escolha as que respondem sua pergunta

---
## EMERGENCIA — quando travar na prova

In [ ]:
# Nao lembro os nomes das colunas
print(df_posts.columns.tolist())

# Ver tipo de dado (object = texto, int/float = numero)
print(df_posts.dtypes)

# Ver valores unicos de uma coluna
print(df_posts['author'].unique())
print(df_posts['author'].value_counts())

# Filtrar linhas
# df[df['coluna'] > 0]
# df[df['coluna'] == 'valor']
# df[df['coluna'].str.contains('texto', na=False)]

# Criar coluna nova
# df['nova'] = df['coluna1'] / df['coluna2']

# Remover linhas nulas em coluna especifica
# df = df.dropna(subset=['coluna_obrigatoria'])

# Preencher nulo com valor padrao
# df['coluna'] = df['coluna'].fillna(0)
# df['coluna'] = df['coluna'].fillna('')

# Verificar se scraping funcionou
# print(resposta.status_code)    # 200 = ok
# print(len(html))               # tamanho do HTML

# Erro no split/treino: coluna com texto? Converte para dummy
# X = pd.get_dummies(X, columns=['coluna_texto'], drop_first=True)

# Erro no KMeans: dado faltando? Verifica
# print(X_cl.isna().sum())

# Silhueta — interpretacao rapida
# > 0.7 = grupos muito bem separados
# 0.5 a 0.7 = razoavelmente separados
# 0.3 a 0.5 = estrutura fraca
# < 0.3 = nao ha estrutura clara

print("Todos os comandos de emergencia acima.")